# 04 · 只补齐 index_constituents

本 notebook 专门用于补齐 `data/raw/index_constituents.parquet` 所需的历史指数成分股和权重。

目标 schema：

```text
date, index_code, code, weight
```

重要说明：

- Choice 没权限时，优先尝试 AkShare / 中证指数官网 / Tushare / 手动 CSV。
- AkShare 的 `index_stock_cons_weight_csindex` 通常只能拿“最新样本权重”，不是完整历史月末权重。
- 如果只能拿最新权重，本 notebook 会保存为 `snapshot_quality = latest_only`，不能伪装成严格历史数据。
- 严格复现需要 `snapshot_quality = historical_weight` 的月末历史成分和权重。
- 默认只写候选文件，不覆盖正式 `index_constituents.parquet`。

输出：

- 分片目录：`data/raw/index_constituents_parts_alt/`
- 候选文件：`data/raw/index_constituents_candidate.parquet`
- 失败表：`data/raw/index_constituents_fetch_failures.csv`
- 覆盖报告：`data/raw/index_constituents_coverage_report.csv`


## 0. 环境、参数和配置

默认不覆盖正式文件。若只想先测试接口，把 `TEST_MODE=True`，只抓前几个指数。

In [44]:
import os
import sys
import time
import json
import shutil
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime

import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RAW = PROJECT_ROOT / "data" / "raw"
CONFIG = PROJECT_ROOT / "config"
PARTS = RAW / "index_constituents_parts_alt"
PARTS.mkdir(parents=True, exist_ok=True)

START_DATE = "2014-01-01"
END_DATE = "2026-04-17"

# 安全参数
OVERWRITE_CANONICAL = True
TEST_MODE = False          # 先用 True 测试；正式抓取改 False
TEST_N_INDEX = 3
SLEEP_SECONDS = 0.5
RETRY = 3

# 指数池参数
USE_STATIC_INDEX_POOL = True
USE_DYNAMIC_POOL_FROM_ETF = True

# 数据源开关
RUN_AKSHARE_LATEST_WEIGHT = True       # 最新权重，非严格历史
RUN_AKSHARE_LATEST_CONS_FALLBACK = True
RUN_TUSHARE_HISTORICAL_WEIGHT = True  # 有 token 且接口权限够时改 True
RUN_MANUAL_CSV_IMPORT = True           # 支持手动导出的历史权重 CSV

TUSHARE_TOKEN = "0467380aa78a5717bf713dc7c4046b8d8b679e63a572d1b58aa70ed7"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW =", RAW)
print("PARTS =", PARTS)


PROJECT_ROOT = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3
RAW = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw
PARTS = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_parts_alt


In [26]:
import yaml

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

strategy_cfg = load_yaml(CONFIG / "strategy.yaml")
index_cfg = load_yaml(CONFIG / "index_universe.yaml")

START_DATE = strategy_cfg.get("backtest", {}).get("data_lookback_start", START_DATE)
END_DATE = strategy_cfg.get("backtest", {}).get("end_date", END_DATE)

print("START_DATE", START_DATE, "END_DATE", END_DATE)
print("static pool size", len(index_cfg.get("static_pool", [])))


START_DATE 2014-01-01 END_DATE 2026-04-17
static pool size 18


In [27]:
def optional_import(name):
    try:
        return __import__(name), None
    except Exception as exc:
        return None, repr(exc)

ak, ak_err = optional_import("akshare")
ts, ts_err = optional_import("tushare")
requests, requests_err = optional_import("requests")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

pd.DataFrame([
    {"package": "akshare", "available": ak is not None, "error": ak_err, "version": getattr(ak, "__version__", None)},
    {"package": "tushare", "available": ts is not None, "error": ts_err, "token": bool(TUSHARE_TOKEN)},
    {"package": "requests", "available": requests is not None, "error": requests_err},
])


,package,available,error,version,token
0,akshare,True,None,1.18.64,NaN
1,tushare,True,None,NaN,True
2,requests,True,None,NaN,NaN


## 1. 指数池生成

优先读取 `config/index_universe.yaml` 的静态宽基池；可选从 `etf_info.tracking_index` 扩展。

In [28]:
def normalize_index_code(code):
    s = str(code).strip().upper().replace(" ", "")
    if not s or s.lower() == "nan":
        return None
    if "." in s:
        raw, exch = s.split(".", 1)
        raw = raw.zfill(6) if raw.isdigit() else raw
        return f"{raw}.{exch}"
    if s.endswith(("SH", "SZ", "CSI", "BJ")):
        # 兼容 000300SH 这类格式。
        if s.endswith("CSI"):
            return f"{s[:-3].zfill(6)}.CSI"
        return f"{s[:-2].zfill(6)}.{s[-2:]}"
    raw = s[:6].zfill(6)
    exch = "SH" if raw.startswith(("000", "0000")) else "SZ"
    return f"{raw}.{exch}"

def bare_index_code(code):
    norm = normalize_index_code(code)
    return norm.split(".")[0] if norm else None

def normalize_stock_code(code):
    s = str(code).strip().upper().replace(" ", "")
    if not s or s.lower() == "nan":
        return None
    if "." in s:
        raw, exch = s.split(".", 1)
        return f"{raw.zfill(6)}.{exch}"
    raw = ''.join(ch for ch in s if ch.isdigit())[:6].zfill(6)
    exch = "SH" if raw.startswith(("6", "9")) else "BJ" if raw.startswith(("8", "4")) else "SZ"
    return f"{raw}.{exch}"

def static_index_pool():
    rows = []
    for x in index_cfg.get("static_pool", []):
        code = normalize_index_code(x.get("code"))
        if code:
            rows.append({"index_code": code, "name": x.get("name"), "pool_source": "config_static"})
    return pd.DataFrame(rows)

def dynamic_index_pool_from_etf():
    path = RAW / "etf_info.parquet"
    if not path.exists():
        return pd.DataFrame(columns=["index_code", "name", "pool_source"])
    df = pd.read_parquet(path)
    if "tracking_index" not in df.columns:
        return pd.DataFrame(columns=["index_code", "name", "pool_source"])
    rows = []
    for v in df["tracking_index"].dropna().astype(str).unique():
        code = normalize_index_code(v)
        if code:
            rows.append({"index_code": code, "name": None, "pool_source": "etf_tracking_index"})
    return pd.DataFrame(rows)

pool_frames = []
if USE_STATIC_INDEX_POOL:
    pool_frames.append(static_index_pool())
if USE_DYNAMIC_POOL_FROM_ETF:
    pool_frames.append(dynamic_index_pool_from_etf())

index_pool_df = pd.concat(pool_frames, ignore_index=True) if pool_frames else pd.DataFrame(columns=["index_code", "name", "pool_source"])
index_pool_df = index_pool_df.drop_duplicates("index_code").sort_values("index_code").reset_index(drop=True)
if TEST_MODE:
    index_pool_df = index_pool_df.head(TEST_N_INDEX).copy()

index_pool = index_pool_df["index_code"].tolist()
print("index count", len(index_pool), "TEST_MODE", TEST_MODE)
index_pool_df


index count 18 TEST_MODE False


,index_code,name,pool_source
0,000010.SH,上证180,config_static
1,000015.SH,上证红利,config_static
2,000016.SH,上证50,config_static
3,000050.SH,上证50等权,config_static
4,000300.SH,沪深300,config_static
5,000510.CSI,中证A500,config_static
6,000688.SH,科创50,config_static
7,000698.SH,科创100,config_static
8,000852.SH,中证1000,config_static
9,000905.SH,中证500,config_static


## 2. 月末交易日

如果本地有 `trade_calendar.parquet`，使用真实月末交易日；否则使用自然月末。

In [29]:
def month_end_dates(start=START_DATE, end=END_DATE):
    cal_path = RAW / "trade_calendar.parquet"
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    if cal_path.exists():
        cal = pd.read_parquet(cal_path)
        date_col = "date" if "date" in cal.columns else "trade_date"
        d = pd.to_datetime(cal[date_col], errors="coerce").dropna().sort_values()
        d = d[(d >= start_ts) & (d <= end_ts)]
        if len(d):
            return pd.Series(d).groupby([d.dt.year, d.dt.month]).max().dt.strftime("%Y-%m-%d").tolist()
    return pd.date_range(start_ts, end_ts, freq="M").strftime("%Y-%m-%d").tolist()

MONTH_ENDS = month_end_dates()
print(len(MONTH_ENDS), MONTH_ENDS[:3], MONTH_ENDS[-3:])


148 ['2014-01-30', '2014-02-28', '2014-03-31'] ['2026-02-27', '2026-03-31', '2026-04-17']


## 3. 字段标准化工具

不同来源列名不稳定，这里用宽松匹配统一成 `date, index_code, code, weight`，并保留 `source`、`snapshot_quality`。

In [30]:
def pick_col(df, candidates, contains=None):
    cols = list(df.columns)
    for c in candidates:
        if c in cols:
            return c
    if contains:
        for col in cols:
            text = str(col).lower()
            if all(k.lower() in text for k in contains):
                return col
    return None

def sanitize_filename(name):
    return str(name).replace("/", "_").replace("\\", "_").replace(":", "_").replace("*", "_").replace("?", "_").replace('"', "_").replace("<", "_").replace(">", "_").replace("|", "_").replace(" ", "")

def normalize_constituent_frame(df, index_code, date, source, snapshot_quality, allow_missing_weight=False):
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["date", "index_code", "code", "weight", "source", "snapshot_quality"])
    raw = df.copy()
    code_col = pick_col(raw, ["code", "证券代码", "成分券代码", "成份券代码", "成分股代码", "品种代码", "con_code", "ts_code", "股票代码"], contains=["代码"])
    weight_col = pick_col(raw, ["weight", "权重", "权重(%)", "权重（%）", "指数权重", "con_weight", "WEIGHT"], contains=["权重"])
    if code_col is None:
        raise RuntimeError(f"cannot find constituent code column, columns={raw.columns.tolist()}")
    out = pd.DataFrame()
    out["date"] = pd.to_datetime(date).strftime("%Y-%m-%d")
    out["index_code"] = normalize_index_code(index_code)
    out["code"] = raw[code_col].map(normalize_stock_code)
    if weight_col is None:
        if allow_missing_weight:
            out["weight"] = pd.NA
        else:
            raise RuntimeError(f"cannot find weight column, columns={raw.columns.tolist()}")
    else:
        w = raw[weight_col].astype(str).str.replace("%", "", regex=False).str.replace(",", "", regex=False)
        out["weight"] = pd.to_numeric(w, errors="coerce")
        # 若权重看起来是小数比例，统一转成百分比。
        if out["weight"].dropna().between(0, 1).mean() > 0.8:
            out["weight"] = out["weight"] * 100
    out["source"] = source
    out["snapshot_quality"] = snapshot_quality
    out = out.dropna(subset=["code"]).drop_duplicates(["date", "index_code", "code"], keep="last")
    return out[["date", "index_code", "code", "weight", "source", "snapshot_quality"]]

def retry_call(fn, *args, retry=RETRY, sleep=SLEEP_SECONDS, **kwargs):
    last = None
    for i in range(1, retry + 1):
        try:
            return fn(*args, **kwargs), None
        except Exception as exc:
            last = exc
            time.sleep(sleep * i)
    return None, repr(last)


## 4. 数据源 A：AkShare 最新权重/成分

`ak.index_stock_cons_weight_csindex(symbol)` 来源是中证指数网站样本权重。它通常只返回最新权重，因此本节保存为一个日期为 `END_DATE` 的最新快照，并标记 `latest_only`。

这可以帮助做接口验证或临时近似，但不能替代 2014-2026 历史月末权重。

In [22]:
def fetch_akshare_latest_weight(index_code):
    if ak is None:
        raise RuntimeError("akshare not installed")
    symbol = bare_index_code(index_code)
    df = ak.index_stock_cons_weight_csindex(symbol=symbol)
    return normalize_constituent_frame(
        df,
        index_code=index_code,
        date=END_DATE,
        source="akshare.index_stock_cons_weight_csindex",
        snapshot_quality="latest_only",
        allow_missing_weight=False,
    )

def fetch_akshare_latest_cons(index_code):
    if ak is None:
        raise RuntimeError("akshare not installed")
    symbol = bare_index_code(index_code)
    try:
        df = ak.index_stock_cons_csindex(symbol=symbol)
        source = "akshare.index_stock_cons_csindex"
    except Exception:
        df = ak.index_stock_cons(symbol=symbol)
        source = "akshare.index_stock_cons"
    return normalize_constituent_frame(
        df,
        index_code=index_code,
        date=END_DATE,
        source=source,
        snapshot_quality="latest_constituents_no_weight",
        allow_missing_weight=True,
    )

def run_akshare_latest_fetch(indices):
    failures = []
    for idx in tqdm(indices, desc="akshare latest"):
        part_path = PARTS / f"ak_latest_{sanitize_filename(idx)}_{END_DATE}.parquet"
        if part_path.exists():
            continue
        df = None
        err = None
        if RUN_AKSHARE_LATEST_WEIGHT:
            df, err = retry_call(fetch_akshare_latest_weight, idx)
        if (df is None or len(df) == 0) and RUN_AKSHARE_LATEST_CONS_FALLBACK:
            df, err2 = retry_call(fetch_akshare_latest_cons, idx)
            err = err or err2
        if df is not None and len(df):
            df.to_parquet(part_path, index=False, engine="pyarrow")
        else:
            failures.append({"index_code": idx, "date": END_DATE, "source": "akshare", "error": err})
        time.sleep(SLEEP_SECONDS)
    return pd.DataFrame(failures)

akshare_failures = run_akshare_latest_fetch(index_pool) if (RUN_AKSHARE_LATEST_WEIGHT or RUN_AKSHARE_LATEST_CONS_FALLBACK) else pd.DataFrame()
akshare_failures


akshare latest: 100%|██████████| 18/18 [00:31<00:00,  1.77s/it]


""


## 5. 数据源 B：Tushare 历史指数权重

如果你有 Tushare Pro token 且具备 `index_weight` 权限，把 `RUN_TUSHARE_HISTORICAL_WEIGHT=True`。这一节会按月末交易日抓取历史权重，质量标记为 `historical_weight`。

In [33]:
def fetch_tushare_index_weight(index_code, date):
    print("fetch", index_code, date, flush=True)
    
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    pro = ts.pro_api(TUSHARE_TOKEN)
    idx = normalize_index_code(index_code)
    ts_idx = idx
    # Tushare 通常使用 000300.SH / 399300.SZ 这类代码；CSI 后缀不一定支持。
    if idx.endswith(".CSI"):
        ts_idx = idx.replace(".CSI", ".SH")
    ts_date = pd.Timestamp(date)
    start = ts_date.replace(day=1).strftime("%Y%m%d")
    end = ts_date.strftime("%Y%m%d")
    df = pro.index_weight(index_code=ts_idx, start_date=start, end_date=end)
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "index_code", "code", "weight", "source", "snapshot_quality"])
    # 若当月有多天权重，取不晚于月末的最后一天。
    date_col = pick_col(df, ["trade_date", "date"])
    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        last_date = df[date_col].max()
        df = df[df[date_col].eq(last_date)]
    return normalize_constituent_frame(
        df,
        index_code=index_code,
        date=date,
        source="tushare.index_weight",
        snapshot_quality="historical_weight",
        allow_missing_weight=False,
    )

def run_tushare_historical_fetch(indices, dates):
    failures = []
    for idx in tqdm(indices, desc="tushare index"):
        for d in dates:
            part_path = PARTS / f"tushare_{sanitize_filename(idx)}_{d}.parquet"
            if part_path.exists():
                continue
            df, err = retry_call(fetch_tushare_index_weight, idx, d)
            if df is not None and len(df):
                df.to_parquet(part_path, index=False, engine="pyarrow")
            else:
                failures.append({"index_code": idx, "date": d, "source": "tushare", "error": err or "empty"})
            time.sleep(SLEEP_SECONDS)
    return pd.DataFrame(failures)

if RUN_TUSHARE_HISTORICAL_WEIGHT:
    tushare_failures = run_tushare_historical_fetch(index_pool, MONTH_ENDS)
else:
    print("RUN_TUSHARE_HISTORICAL_WEIGHT=False，未执行 Tushare 历史权重抓取。")
    tushare_failures = pd.DataFrame()

tushare_failures.head()


tushare index:   0%|          | 0/18 [00:00<?, ?it/s]

fetch 000010.SH 2026-04-17


tushare index:   6%|▌         | 1/18 [00:02<00:44,  2.59s/it]

fetch 000015.SH 2026-04-17


tushare index:  11%|█         | 2/18 [00:03<00:22,  1.40s/it]

...

fetch 899050.BJ 2014-01-30
fetch 899050.BJ 2014-02-28
fetch 899050.BJ 2014-03-31
fetch 899050.BJ 2014-04-30
fetch 899050.BJ 2014-05-30
fetch 899050.BJ 2014-06-30
fetch 899050.BJ 2014-07-31
fetch 899050.BJ 2014-08-29
fetch 899050.BJ 2014-09-30
fetch 899050.BJ 2014-10-31
fetch 899050.BJ 2014-11-28
fetch 899050.BJ 2014-12-31
fetch 899050.BJ 2015-01-30
fetch 899050.BJ 2015-02-27
fetch 899050.BJ 2015-03-31
fetch 899050.BJ 2015-04-30
fetch 899050.BJ 2015-05-29
fetch 899050.BJ 2015-06-30
fetch 899050.BJ 2015-07-31
fetch 899050.BJ 2015-08-31
fetch 899050.BJ 2015-09-30
fetch 899050.BJ 2015-10-30
fetch 899050.BJ 2015-11-30
fetch 899050.BJ 2015-12-31
fetch 899050.BJ 2016-01-29
fetch 899050.BJ 2016-02-29
fetch 899050.BJ 2016-03-31
fetch 899050.BJ 2016-04-29
fetch 899050.BJ 2016-05-31
fetch 899050.BJ 2016-06-30
fetch 899050.BJ 2016-07-29
fetch 899050.BJ 2016-08-31
fetch 899050.BJ 2016-09-30
fetch 899050.BJ 2016-10-31
fetch 899050.BJ 2016-11-30
fetch 899050.BJ 2016-12-30
fetch 899050.BJ 2017-01-26
f

tushare index:  94%|█████████▍| 17/18 [09:42<01:09, 69.40s/it]

fetch 932000.CSI 2014-01-30
fetch 932000.CSI 2014-02-28
fetch 932000.CSI 2014-03-31
fetch 932000.CSI 2014-04-30
fetch 932000.CSI 2014-05-30
fetch 932000.CSI 2014-06-30
fetch 932000.CSI 2014-07-31
fetch 932000.CSI 2014-08-29
fetch 932000.CSI 2014-09-30
fetch 932000.CSI 2014-10-31
fetch 932000.CSI 2014-11-28
fetch 932000.CSI 2014-12-31
fetch 932000.CSI 2015-01-30
fetch 932000.CSI 2015-02-27
fetch 932000.CSI 2015-03-31
fetch 932000.CSI 2015-04-30
fetch 932000.CSI 2015-05-29
fetch 932000.CSI 2015-06-30
fetch 932000.CSI 2015-07-31
fetch 932000.CSI 2015-08-31
fetch 932000.CSI 2015-09-30
fetch 932000.CSI 2015-10-30
fetch 932000.CSI 2015-11-30
fetch 932000.CSI 2015-12-31
fetch 932000.CSI 2016-01-29
fetch 932000.CSI 2016-02-29
fetch 932000.CSI 2016-03-31
fetch 932000.CSI 2016-04-29
fetch 932000.CSI 2016-05-31
fetch 932000.CSI 2016-06-30
fetch 932000.CSI 2016-07-29
fetch 932000.CSI 2016-08-31
fetch 932000.CSI 2016-09-30
fetch 932000.CSI 2016-10-31
fetch 932000.CSI 2016-11-30
fetch 932000.CSI 201

tushare index: 100%|██████████| 18/18 [11:11<00:00, 37.31s/it]


,index_code,date,source,error
0,000010.SH,2026-04-17,tushare,empty
1,000015.SH,2026-04-17,tushare,empty
2,000016.SH,2026-04-17,tushare,empty
3,000050.SH,2026-04-17,tushare,empty
4,000300.SH,2014-01-30,tushare,empty


In [34]:
from pathlib import Path
import re
import pandas as pd
import duckdb

parts_dir = PARTS
files = sorted(parts_dir.glob("tushare_*.parquet"))

print("Tushare 分片数量:", len(files))
print("目录:", parts_dir)

rows = []

for p in files:
    rec = {
        "file": p.name,
        "path": str(p),
        "size": p.stat().st_size,
    }

    m = re.match(r"tushare_(.+)_(\d{4}-\d{2}-\d{2})\.parquet$", p.name)
    if m:
        rec["index_code_from_file"] = m.group(1)
        rec["date_from_file"] = m.group(2)

    try:
        q = str(p).replace("'", "''")
        df = duckdb.sql(f"select * from read_parquet('{q}')").df()

        rec["readable"] = True
        rec["rows"] = len(df)
        rec["columns"] = ",".join(df.columns)

        for c in ["date", "index_code", "code", "weight", "source", "snapshot_quality"]:
            rec[f"has_{c}"] = c in df.columns

        if len(df):
            if "date" in df.columns:
                rec["date_min"] = pd.to_datetime(df["date"], errors="coerce").min()
                rec["date_max"] = pd.to_datetime(df["date"], errors="coerce").max()
            if "index_code" in df.columns:
                rec["index_code_nunique"] = df["index_code"].nunique(dropna=True)
                rec["index_code_sample"] = ",".join(df["index_code"].dropna().astype(str).unique()[:3])
            if "code" in df.columns:
                rec["stock_n"] = df["code"].nunique(dropna=True)
            if "weight" in df.columns:
                w = pd.to_numeric(df["weight"], errors="coerce")
                rec["weight_nonnull_rate"] = w.notna().mean()
                rec["weight_sum"] = w.sum()
                rec["weight_min"] = w.min()
                rec["weight_max"] = w.max()

    except Exception as e:
        rec["readable"] = False
        rec["error"] = repr(e)

    rows.append(rec)

audit = pd.DataFrame(rows)

display_cols = [
    "file", "readable", "rows", "index_code_from_file", "date_from_file",
    "stock_n", "weight_nonnull_rate", "weight_sum", "size"
]
display(audit[display_cols].head(20))

print("\n=== 总体 ===")
print("文件数:", len(audit))
print("可读文件数:", audit["readable"].sum())
print("不可读文件数:", (~audit["readable"]).sum())
print("总行数:", audit.loc[audit["readable"], "rows"].sum())
print("0行文件数:", (audit["rows"].fillna(-1) == 0).sum())

print("\n=== 按指数覆盖 ===")
by_index = (
    audit[audit["readable"]]
    .groupby("index_code_from_file", dropna=False)
    .agg(
        files=("file", "count"),
        rows=("rows", "sum"),
        min_date=("date_from_file", "min"),
        max_date=("date_from_file", "max"),
        min_weight_sum=("weight_sum", "min"),
        max_weight_sum=("weight_sum", "max"),
        min_stock_n=("stock_n", "min"),
        max_stock_n=("stock_n", "max"),
    )
    .sort_index()
)
display(by_index)

print("\n=== 异常：不可读 ===")
bad_read = audit[~audit["readable"]]
display(bad_read[["file", "error"]].head(50))

print("\n=== 异常：0 行 ===")
zero = audit[audit["rows"].fillna(-1) == 0]
display(zero[display_cols].head(50))

print("\n=== 异常：缺关键字段 ===")
required_cols = ["date", "index_code", "code", "weight", "source", "snapshot_quality"]
missing_mask = False
for c in required_cols:
    missing_mask = missing_mask | (audit.get(f"has_{c}") != True)
missing_cols = audit[audit["readable"] & missing_mask]
display(missing_cols[["file"] + [f"has_{c}" for c in required_cols]].head(50))

print("\n=== 异常：权重缺失或权重和可疑 ===")
weight_bad = audit[
    audit["readable"]
    & (
        (audit["weight_nonnull_rate"].fillna(0) < 1)
        | (audit["weight_sum"].fillna(0) <= 0)
        | (audit["weight_sum"].fillna(0) > 105)
    )
]
display(weight_bad[display_cols].head(100))

print("\n=== 最近 20 个文件 ===")
recent = audit.copy()
recent["mtime"] = [Path(p).stat().st_mtime for p in recent["path"]]
recent = recent.sort_values("mtime").tail(20)
display(recent[display_cols + ["mtime"]])

audit_path = parts_dir / "tushare_parts_audit.csv"
audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
print("\n已保存审计结果:", audit_path)

Tushare 分片数量: 1912
目录: /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_parts_alt


,file,readable,rows,index_code_from_file,date_from_file,stock_n,weight_nonnull_rate,weight_sum,size
0,tushare_000010.SH_2014-01-30.parquet,True,180,000010.SH,2014-01-30,180,1.0,10000.0,5526
1,tushare_000010.SH_2014-02-28.parquet,True,180,000010.SH,2014-02-28,180,1.0,10001.0,5524
2,tushare_000010.SH_2014-03-31.parquet,True,180,000010.SH,2014-03-31,180,1.0,9995.0,5541
3,tushare_000010.SH_2014-04-30.parquet,True,180,000010.SH,2014-04-30,180,1.0,9994.0,5520
4,tushare_000010.SH_2014-05-30.parquet,True,180,000010.SH,2014-05-30,180,1.0,9999.0,5519
5,tushare_000010.SH_2014-06-30.parquet,True,180,000010.SH,2014-06-30,180,1.0,9996.0,5526
6,tushare_000010.SH_2014-07-31.parquet,True,180,000010.SH,2014-07-31,180,1.0,9992.0,5529
7,tushare_000010.SH_2014-08-29.parquet,True,180,000010.SH,2014-08-29,180,1.0,10004.0,5520
8,tushare_000010.SH_2014-09-30.parquet,True,180,000010.SH,2014-09-30,180,1.0,9995.0,5508
9,tushare_000010.SH_2014-10-31.parquet,True,180,000010.SH,2014-10-31,180,1.0,9998.0,5530



=== 总体 ===
文件数: 1912
可读文件数: 1912
不可读文件数: 0
总行数: 1001049
0行文件数: 0

=== 按指数覆盖 ===


,files,rows,min_date,max_date,min_weight_sum,max_weight_sum,min_stock_n,max_stock_n
index_code_from_file,,,,,,,,
000010.SH,147,26460,2014-01-30,2026-03-31,9992.000,10004.000,180,180
000015.SH,147,7350,2014-01-30,2026-03-31,99.970,100.050,50,50
000016.SH,147,7350,2014-01-30,2026-03-31,99.990,100.070,50,50
000050.SH,147,7350,2014-01-30,2026-03-31,99.960,100.030,50,50
000300.SH,124,37200,2016-01-29,2026-04-17,9998.800,10001.100,300,300
000510.CSI,18,9000,2024-10-31,2026-03-31,9999.000,10001.300,500,500
000688.SH,69,3450,2020-07-31,2026-03-31,99.996,100.005,50,50
000698.SH,32,3200,2023-08-31,2026-03-31,99.997,100.006,100,100
000852.SH,138,131071,2014-10-31,2026-03-31,21.000,10019.600,3,1002



=== 异常：不可读 ===


KeyError: "['error'] not in index"

## 6. 数据源 C：手动 CSV 导入

当免费 API 拿不到历史权重时，可以从中证指数官网、交易所、Wind/Choice 导出 CSV，然后放到：

```text
data/manual/index_constituents/
```

支持两种格式：

1. 单文件已包含字段：`date,index_code,code,weight`
2. 文件名包含指数和日期，例如 `000300.SH_2020-12-31.csv`，文件内包含股票代码和权重列

导入后统一保存为分片，质量标记为 `historical_weight_manual`。

In [ ]:
MANUAL_DIR = PROJECT_ROOT / "data" / "manual" / "index_constituents"
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

KNOWN_ENCODINGS = ["utf-8-sig", "utf-8", "gbk", "gb18030"]

def read_csv_flexible(path):
    last = None
    for enc in KNOWN_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc), None
        except Exception as exc:
            last = exc
    return None, repr(last)

def parse_manual_filename(path):
    stem = Path(path).stem
    parts = stem.replace("_", "-").split("-")
    # 宽松解析：优先找 6 位指数代码和 yyyy-mm-dd。
    idx = None
    for token in stem.replace("_", " ").split():
        if any(ch.isdigit() for ch in token) and len(''.join(ch for ch in token if ch.isdigit())) >= 6:
            idx = normalize_index_code(token)
            break
    import re
    m = re.search(r"(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d)", stem)
    date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}" if m else None
    return idx, date

def import_manual_constituents_csv(path):
    raw, err = read_csv_flexible(path)
    if err:
        raise RuntimeError(err)
    idx_from_name, date_from_name = parse_manual_filename(path)
    date_col = pick_col(raw, ["date", "日期", "调整日期", "生效日期", "trade_date"])
    index_col = pick_col(raw, ["index_code", "指数代码", "指数"])
    if date_col:
        date_value = pd.to_datetime(raw[date_col], errors="coerce").dropna().max().strftime("%Y-%m-%d")
    else:
        date_value = date_from_name
    if index_col:
        index_value = normalize_index_code(raw[index_col].dropna().astype(str).iloc[0])
    else:
        index_value = idx_from_name
    if not date_value or not index_value:
        raise RuntimeError("manual csv must include date/index_code columns or filename like 000300.SH_2020-12-31.csv")
    return normalize_constituent_frame(
        raw,
        index_code=index_value,
        date=date_value,
        source=f"manual_csv:{Path(path).name}",
        snapshot_quality="historical_weight_manual",
        allow_missing_weight=False,
    )

def run_manual_import():
    failures = []
    files = sorted(MANUAL_DIR.glob("*.csv"))
    for path in tqdm(files, desc="manual csv"):
        part_path = PARTS / f"manual_{sanitize_filename(path.stem)}.parquet"
        if part_path.exists():
            continue
        try:
            df = import_manual_constituents_csv(path)
            if len(df):
                df.to_parquet(part_path, index=False, engine="pyarrow")
            else:
                failures.append({"file": str(path), "error": "empty after normalize"})
        except Exception as exc:
            failures.append({"file": str(path), "error": repr(exc)})
    return pd.DataFrame(failures)

if RUN_MANUAL_CSV_IMPORT:
    manual_failures = run_manual_import()
else:
    manual_failures = pd.DataFrame()

print("manual dir", MANUAL_DIR)
manual_failures.head()


## 7. 合并分片为候选文件

严格历史权重优先级最高，其次手动历史权重，最后才是 AkShare 最新快照。合并时同一 `date,index_code,code` 保留质量更高的数据。

In [35]:
QUALITY_RANK = {
    "historical_weight": 3,
    "historical_weight_manual": 3,
    "latest_only": 1,
    "latest_constituents_no_weight": 0,
}

def merge_parts():
    frames = []
    bad = []
    for path in sorted(PARTS.glob("*.parquet")):
        try:
            df = pd.read_parquet(path)
            df["part_file"] = path.name
            frames.append(df)
        except Exception as exc:
            bad.append({"part": str(path), "error": repr(exc)})
    if not frames:
        return pd.DataFrame(), pd.DataFrame(bad)
    all_df = pd.concat(frames, ignore_index=True)
    for c in ["date", "index_code", "code", "weight", "source", "snapshot_quality"]:
        if c not in all_df.columns:
            all_df[c] = pd.NA
    all_df["quality_rank"] = all_df["snapshot_quality"].map(QUALITY_RANK).fillna(-1)
    all_df["date"] = pd.to_datetime(all_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    all_df = all_df.dropna(subset=["date", "index_code", "code"])
    all_df = all_df.sort_values(["date", "index_code", "code", "quality_rank"]).drop_duplicates(["date", "index_code", "code"], keep="last")
    all_df = all_df.sort_values(["date", "index_code", "code"]).reset_index(drop=True)
    return all_df, pd.DataFrame(bad)

candidate, bad_parts = merge_parts()
candidate_path = RAW / "index_constituents_candidate.parquet"
if len(candidate):
    candidate[["date", "index_code", "code", "weight", "source", "snapshot_quality"]].to_parquet(candidate_path, index=False, engine="pyarrow")
    print("saved", candidate_path, "rows", len(candidate))
else:
    print("no parts to merge")

bad_parts


no parts to merge


""


## 8. 覆盖率和质量报告

重点看：

- 是否覆盖全部指数池
- 是否覆盖全部月末日期
- `weight` 非空率
- `snapshot_quality` 是否为历史权重


In [36]:
def build_coverage_report(df):
    rows = []
    if df is None or len(df) == 0:
        return pd.DataFrame()
    tmp = df.copy()
    tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
    for idx, g in tmp.groupby("index_code"):
        dates = sorted(g["date"].dropna().dt.strftime("%Y-%m-%d").unique())
        qualities = sorted(g["snapshot_quality"].dropna().astype(str).unique()) if "snapshot_quality" in g.columns else []
        rows.append({
            "index_code": idx,
            "n_dates": len(dates),
            "date_min": dates[0] if dates else None,
            "date_max": dates[-1] if dates else None,
            "n_constituent_rows": len(g),
            "n_unique_stocks": g["code"].nunique(),
            "weight_nonnull_rate": round(g["weight"].notna().mean(), 4),
            "quality": ",".join(qualities),
            "has_historical_weight": any(q in {"historical_weight", "historical_weight_manual"} for q in qualities),
            "has_latest_only": "latest_only" in qualities or "latest_constituents_no_weight" in qualities,
        })
    return pd.DataFrame(rows).sort_values("index_code")

coverage = build_coverage_report(candidate)
coverage_path = RAW / "index_constituents_coverage_report.csv"
coverage.to_csv(coverage_path, index=False, encoding="utf-8-sig")
print("saved", coverage_path)
coverage


saved /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_coverage_report.csv


""


In [37]:
failures = []
for name in ["akshare_failures", "tushare_failures", "manual_failures"]:
    df = globals().get(name)
    if df is not None and len(df):
        x = df.copy()
        x["failure_table"] = name
        failures.append(x)
failures_df = pd.concat(failures, ignore_index=True, sort=False) if failures else pd.DataFrame()
failures_path = RAW / "index_constituents_fetch_failures.csv"
failures_df.to_csv(failures_path, index=False, encoding="utf-8-sig")
print("saved", failures_path, "rows", len(failures_df))
failures_df.head(30)


saved /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_fetch_failures.csv rows 752


,index_code,date,source,error,failure_table
0,000010.SH,2026-04-17,tushare,empty,tushare_failures
1,000015.SH,2026-04-17,tushare,empty,tushare_failures
2,000016.SH,2026-04-17,tushare,empty,tushare_failures
3,000050.SH,2026-04-17,tushare,empty,tushare_failures
4,000300.SH,2014-01-30,tushare,empty,tushare_failures
5,000300.SH,2014-02-28,tushare,empty,tushare_failures
6,000300.SH,2014-03-31,tushare,empty,tushare_failures
7,000300.SH,2014-04-30,tushare,empty,tushare_failures
8,000300.SH,2014-05-30,tushare,empty,tushare_failures
9,000300.SH,2014-06-30,tushare,empty,tushare_failures


## 9. 可选覆盖正式文件

只有当候选文件已经满足以下条件时才建议覆盖：

- `date,index_code,code,weight` 字段完整；
- 覆盖策略所需指数池和 2014-2026 月末；
- `weight` 非空率足够高；
- `snapshot_quality` 主要是 `historical_weight` 或 `historical_weight_manual`。


In [38]:
def candidate_is_safe_for_canonical(df):
    if df is None or len(df) == 0:
        return False, "candidate empty"

    required = ["date", "index_code", "code", "weight"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        return False, f"missing columns: {missing}"

    if df["weight"].notna().mean() < 0.8:
        return False, "weight nonnull rate < 80%"

    dup = df.duplicated(["date", "index_code", "code"]).sum()
    if dup > 0:
        return False, f"duplicate date/index/code rows: {dup}"

    g = df.groupby(["date", "index_code"])
    stock_n = g["code"].nunique()
    weight_sum = g["weight"].sum()

    if (weight_sum <= 0).any():
        return False, "some date/index groups have non-positive weight sum"

    suspicious_small = stock_n[stock_n < 20]
    if len(suspicious_small):
        return False, f"some date/index groups have too few constituents, e.g. {suspicious_small.head().to_dict()}"

    qualities = set(
        df.get("snapshot_quality", pd.Series(dtype=str))
        .dropna()
        .astype(str)
        .unique()
    )
    if not qualities.intersection({"historical_weight", "historical_weight_manual"}):
        return False, "no historical weight data; only latest snapshot is not enough"

    return True, "ok"

In [39]:
ok, reason = candidate_is_safe_for_canonical(candidate)
print("safe_to_overwrite =", ok)
print("reason =", reason)

safe_to_overwrite = False
reason = candidate empty


In [40]:
from pathlib import Path
import duckdb
import pandas as pd

parts = sorted(PARTS.glob("tushare_*.parquet"))
print("parts:", len(parts))

if not parts:
    raise RuntimeError(f"No tushare parts found in {PARTS}")

pattern = str(PARTS / "tushare_*.parquet").replace("'", "''")

candidate = duckdb.sql(f"""
    select *
    from read_parquet('{pattern}', union_by_name=true)
""").df()

print("candidate shape:", candidate.shape)
display(candidate.head())
print(candidate.columns.tolist())

parts: 1912
candidate shape: (1001049, 6)


,date,index_code,code,weight,source,snapshot_quality
0,<NA>,<NA>,601318.SH,550.0,tushare.index_weight,historical_weight
1,<NA>,<NA>,600036.SH,502.0,tushare.index_weight,historical_weight
2,<NA>,<NA>,600016.SH,486.0,tushare.index_weight,historical_weight
3,<NA>,<NA>,601166.SH,314.0,tushare.index_weight,historical_weight
4,<NA>,<NA>,600000.SH,301.0,tushare.index_weight,historical_weight


['date', 'index_code', 'code', 'weight', 'source', 'snapshot_quality']


In [41]:
candidate = candidate.copy()

candidate["date"] = pd.to_datetime(candidate["date"], errors="coerce")
candidate["weight"] = pd.to_numeric(candidate["weight"], errors="coerce")

if "source" not in candidate.columns:
    candidate["source"] = "tushare.index_weight"

if "snapshot_quality" not in candidate.columns:
    candidate["snapshot_quality"] = "historical_weight"

# 去重
candidate = candidate.drop_duplicates(["date", "index_code", "code"])

# 先找异常组
check = (
    candidate
    .groupby(["date", "index_code"])
    .agg(
        stock_n=("code", "nunique"),
        weight_sum=("weight", "sum"),
        weight_nonnull_rate=("weight", lambda s: s.notna().mean()),
    )
    .reset_index()
)

bad = check[
    (check["stock_n"] < 20)
    | (check["weight_sum"] <= 0)
    | (check["weight_nonnull_rate"] < 0.8)
]

display(bad.sort_values(["index_code", "date"]).head(200))
print("异常组数:", len(bad))

,date,index_code,stock_n,weight_sum,weight_nonnull_rate


异常组数: 0


In [42]:
ok, reason = candidate_is_safe_for_canonical(candidate)
print("safe_to_overwrite =", ok)
print("reason =", reason)

safe_to_overwrite = True
reason = ok


In [49]:
def candidate_is_safe_for_canonical(df):
    if df is None or len(df) == 0:
        return False, "candidate empty"
    required = ["date", "index_code", "code", "weight"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        return False, f"missing columns: {missing}"
    if df["weight"].notna().mean() < 0.8:
        return False, "weight nonnull rate < 80%"
    qualities = set(df.get("snapshot_quality", pd.Series(dtype=str)).dropna().astype(str).unique())
    if not qualities.intersection({"historical_weight", "historical_weight_manual"}):
        return False, "no historical weight data; only latest snapshot is not enough"
    return True, "ok"

ok, reason = candidate_is_safe_for_canonical(candidate)
print("safe_to_overwrite =", ok, "reason =", reason)

if OVERWRITE_CANONICAL:
    if not ok:
        raise RuntimeError(f"candidate not safe to overwrite: {reason}")
    canonical = RAW / "index_constituents.parquet"
    backup_dir = RAW / "_backup_before_overwrite"
    backup_dir.mkdir(parents=True, exist_ok=True)
    if canonical.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy2(canonical, backup_dir / f"{canonical.stem}.{stamp}{canonical.suffix}")
    shutil.copy2(candidate_path, canonical)
    print("overwritten", canonical)
else:
    print("OVERWRITE_CANONICAL=False，未覆盖正式文件。")


safe_to_overwrite = True reason = ok
overwritten /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents.parquet


In [51]:
from pathlib import Path
from datetime import datetime
import shutil
import duckdb
import pandas as pd
import numpy as np

PARTS = RAW / "index_constituents_parts_alt"
pattern = str(PARTS / "tushare_*.parquet").replace("'", "''")

# 1. 从所有 Tushare 分片读取，并从 filename 恢复 index_code/date
candidate = duckdb.sql(f"""
    select
        cast(regexp_extract(filename, 'tushare_([^/]+)_([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})\\.parquet$', 2) as date) as date,
        regexp_extract(filename, 'tushare_([^/]+)_([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})\\.parquet$', 1) as index_code,
        code,
        cast(weight as double) as weight,
        coalesce(source, 'tushare.index_weight') as source,
        coalesce(snapshot_quality, 'historical_weight') as snapshot_quality
    from read_parquet('{pattern}', union_by_name=true, filename=true)
    where code is not null
      and weight is not null
""").df()

candidate["date"] = pd.to_datetime(candidate["date"], errors="coerce")
candidate["index_code"] = candidate["index_code"].astype(str)
candidate["code"] = candidate["code"].astype(str)
candidate["weight"] = pd.to_numeric(candidate["weight"], errors="coerce")

print("raw candidate:", candidate.shape)
display(candidate.head())

# 2. 去重
candidate = candidate.drop_duplicates(["date", "index_code", "code"]).copy()

# 3. 找明显异常组
check_raw = (
    candidate
    .groupby(["date", "index_code"])
    .agg(
        stock_n=("code", "nunique"),
        weight_raw_sum=("weight", "sum"),
        weight_nonnull_rate=("weight", lambda s: s.notna().mean()),
    )
    .reset_index()
)

bad = check_raw[
    (check_raw["date"].isna())
    | (check_raw["index_code"].isna())
    | (check_raw["stock_n"] < 20)
    | (check_raw["weight_raw_sum"] <= 0)
    | (check_raw["weight_nonnull_rate"] < 0.8)
].copy()

print("bad groups:", len(bad))
display(bad.sort_values(["index_code", "date"]).head(200))

# 4. 剔除明显残缺组，例如 000852.SH 早期只有 3 个成分
bad_keys = bad[["date", "index_code"]].drop_duplicates()

candidate_clean = (
    candidate
    .merge(bad_keys.assign(_bad=1), on=["date", "index_code"], how="left")
    .query("_bad.isna()")
    .drop(columns="_bad")
    .copy()
)

# 5. 保留原始权重，并归一化到每个 date/index_code 合计为 1
candidate_clean["weight_raw"] = candidate_clean["weight"]
group_sum = candidate_clean.groupby(["date", "index_code"])["weight"].transform("sum")
candidate_clean["weight"] = candidate_clean["weight"] / group_sum

# 6. 最终检查
check = (
    candidate_clean
    .groupby(["date", "index_code"])
    .agg(
        stock_n=("code", "nunique"),
        weight_sum=("weight", "sum"),
        weight_raw_sum=("weight_raw", "sum"),
    )
    .reset_index()
)

display(check.groupby("index_code").agg(
    files=("date", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
    min_stock_n=("stock_n", "min"),
    max_stock_n=("stock_n", "max"),
    min_weight_sum=("weight_sum", "min"),
    max_weight_sum=("weight_sum", "max"),
))

bad_after = check[
    (check["stock_n"] < 20)
    | (check["weight_sum"] < 0.999)
    | (check["weight_sum"] > 1.001)
]

print("bad after clean:", len(bad_after))
display(bad_after.head(100))

candidate = candidate_clean

ok, reason = candidate_is_safe_for_canonical(candidate)
print("safe_to_overwrite =", ok, "reason =", reason)
print("candidate rows:", len(candidate))

if not ok:
    raise RuntimeError(reason)

# 7. 保存正确候选
candidate_path = RAW / "index_constituents_candidate.parquet"
candidate.to_parquet(candidate_path, index=False, engine="pyarrow")

print("saved candidate:", candidate_path)
print("candidate size MB:", candidate_path.stat().st_size / 1024 / 1024)

# 8. 覆盖 canonical，先备份当前坏文件
canonical = RAW / "index_constituents.parquet"
backup_dir = RAW / "_backup_before_overwrite"
backup_dir.mkdir(parents=True, exist_ok=True)

if canonical.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = backup_dir / f"{canonical.stem}.{stamp}{canonical.suffix}"
    shutil.copy2(canonical, backup_path)
    print("backup old canonical:", backup_path)

shutil.copy2(candidate_path, canonical)
print("overwritten:", canonical)

raw candidate: (1001049, 6)


,date,index_code,code,weight,source,snapshot_quality
0,2014-01-30,000010.SH,601318.SH,550.0,tushare.index_weight,historical_weight
1,2014-01-30,000010.SH,600036.SH,502.0,tushare.index_weight,historical_weight
2,2014-01-30,000010.SH,600016.SH,486.0,tushare.index_weight,historical_weight
3,2014-01-30,000010.SH,601166.SH,314.0,tushare.index_weight,historical_weight
4,2014-01-30,000010.SH,600000.SH,301.0,tushare.index_weight,historical_weight


bad groups: 7


,date,index_code,stock_n,weight_raw_sum,weight_nonnull_rate
98,2014-10-31,000852.SH,3,23.0,1.0
110,2014-11-28,000852.SH,3,23.0,1.0
122,2014-12-31,000852.SH,3,26.0,1.0
134,2015-01-30,000852.SH,3,22.0,1.0
146,2015-02-27,000852.SH,3,23.0,1.0
158,2015-03-31,000852.SH,3,22.0,1.0
170,2015-04-30,000852.SH,3,21.0,1.0


,files,min_date,max_date,min_stock_n,max_stock_n,min_weight_sum,max_weight_sum
index_code,,,,,,,
000010.SH,147,2014-01-30,2026-03-31,180,180,1.0,1.0
000015.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
000016.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
000050.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
000300.SH,124,2016-01-29,2026-04-17,300,300,1.0,1.0
000510.CSI,18,2024-10-31,2026-03-31,500,500,1.0,1.0
000688.SH,69,2020-07-31,2026-03-31,50,50,1.0,1.0
000698.SH,32,2023-08-31,2026-03-31,100,100,1.0,1.0
000852.SH,131,2015-05-29,2026-03-31,1000,1002,1.0,1.0


bad after clean: 0


,date,index_code,stock_n,weight_sum,weight_raw_sum


safe_to_overwrite = True reason = ok
candidate rows: 1001028
saved candidate: /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_candidate.parquet
candidate size MB: 3.822157859802246
backup old canonical: /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/_backup_before_overwrite/index_constituents.20260612_215241.parquet
overwritten: /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents.parquet


In [52]:
canonical = RAW / "index_constituents.parquet"
q = str(canonical).replace("'", "''")

summary = duckdb.sql(f"""
    select
        count(*) as rows,
        count(distinct index_code) as n_index,
        min(date) as min_date,
        max(date) as max_date
    from read_parquet('{q}')
""").df()

display(summary)

weight_check = duckdb.sql(f"""
    select
        index_code,
        count(distinct date) as n_dates,
        min(date) as min_date,
        max(date) as max_date,
        min(stock_n) as min_stock_n,
        max(stock_n) as max_stock_n,
        min(weight_sum) as min_weight_sum,
        max(weight_sum) as max_weight_sum
    from (
        select
            date,
            index_code,
            count(distinct code) as stock_n,
            sum(weight) as weight_sum
        from read_parquet('{q}')
        group by date, index_code
    )
    group by index_code
    order by index_code
""").df()

display(weight_check)

,rows,n_index,min_date,max_date
0,1001028,17,2014-01-30,2026-04-17


,index_code,n_dates,min_date,max_date,min_stock_n,max_stock_n,min_weight_sum,max_weight_sum
0,000010.SH,147,2014-01-30,2026-03-31,180,180,1.0,1.0
1,000015.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
2,000016.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
3,000050.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
4,000300.SH,124,2016-01-29,2026-04-17,300,300,1.0,1.0
5,000510.CSI,18,2024-10-31,2026-03-31,500,500,1.0,1.0
6,000688.SH,69,2020-07-31,2026-03-31,50,50,1.0,1.0
7,000698.SH,32,2023-08-31,2026-03-31,100,100,1.0,1.0
8,000852.SH,131,2015-05-29,2026-03-31,1000,1002,1.0,1.0
9,000905.SH,147,2014-01-30,2026-03-31,500,500,1.0,1.0


In [53]:
check = (
    candidate
    .groupby(["date", "index_code"])
    .agg(
        stock_n=("code", "nunique"),
        rows=("code", "size"),
        weight_sum=("weight", "sum"),
    )
    .reset_index()
)

display(check.sort_values("stock_n").head(50))

,date,index_code,stock_n,rows,weight_sum
1650,2024-12-31,000016.SH,50,50,1.0
1287,2022-11-30,000016.SH,50,50,1.0
876,2020-01-23,000050.SH,50,50,1.0
486,2017-05-31,399673.SZ,50,50,1.0
1574,2024-07-31,000050.SH,50,50,1.0
488,2017-06-30,000015.SH,50,50,1.0
489,2017-06-30,000016.SH,50,50,1.0
490,2017-06-30,000050.SH,50,50,1.0
1573,2024-07-31,000016.SH,50,50,1.0
1576,2024-07-31,000688.SH,50,50,1.0


In [54]:
display(candidate.head(20))
print(candidate.shape)
print(candidate[["date", "index_code", "code", "weight"]].head(20))
print(candidate["code"].nunique())

,date,index_code,code,weight,source,snapshot_quality,weight_raw
0,2014-01-30,000010.SH,601318.SH,0.0550,tushare.index_weight,historical_weight,550.0
1,2014-01-30,000010.SH,600036.SH,0.0502,tushare.index_weight,historical_weight,502.0
2,2014-01-30,000010.SH,600016.SH,0.0486,tushare.index_weight,historical_weight,486.0
3,2014-01-30,000010.SH,601166.SH,0.0314,tushare.index_weight,historical_weight,314.0
4,2014-01-30,000010.SH,600000.SH,0.0301,tushare.index_weight,historical_weight,301.0
5,2014-01-30,000010.SH,600837.SH,0.0244,tushare.index_weight,historical_weight,244.0
6,2014-01-30,000010.SH,600030.SH,0.0234,tushare.index_weight,historical_weight,234.0
7,2014-01-30,000010.SH,601288.SH,0.0181,tushare.index_weight,historical_weight,181.0
8,2014-01-30,000010.SH,601328.SH,0.0174,tushare.index_weight,historical_weight,174.0
9,2014-01-30,000010.SH,601398.SH,0.0163,tushare.index_weight,historical_weight,163.0


(1001028, 7)
         date index_code       code  weight
0  2014-01-30  000010.SH  601318.SH  0.0550
1  2014-01-30  000010.SH  600036.SH  0.0502
2  2014-01-30  000010.SH  600016.SH  0.0486
3  2014-01-30  000010.SH  601166.SH  0.0314
4  2014-01-30  000010.SH  600000.SH  0.0301
5  2014-01-30  000010.SH  600837.SH  0.0244
6  2014-01-30  000010.SH  600030.SH  0.0234
7  2014-01-30  000010.SH  601288.SH  0.0181
8  2014-01-30  000010.SH  601328.SH  0.0174
9  2014-01-30  000010.SH  601398.SH  0.0163
10 2014-01-30  000010.SH  600519.SH  0.0161
11 2014-01-30  000010.SH  601601.SH  0.0151
12 2014-01-30  000010.SH  600887.SH  0.0149
13 2014-01-30  000010.SH  601088.SH  0.0134
14 2014-01-30  000010.SH  601668.SH  0.0132
15 2014-01-30  000010.SH  601006.SH  0.0127
16 2014-01-30  000010.SH  600104.SH  0.0126
17 2014-01-30  000010.SH  601818.SH  0.0118
18 2014-01-30  000010.SH  601939.SH  0.0111
19 2014-01-30  000010.SH  601169.SH  0.0110
5812


In [55]:
check = (
    candidate
    .groupby(["date", "index_code"], as_index=False)
    .agg(
        stock_n=("code", "nunique"),
        rows=("code", "size"),
        weight_sum=("weight", "sum"),
    )
)

display(check.sort_values("stock_n").head(30))

,date,index_code,stock_n,rows,weight_sum
1650,2024-12-31,000016.SH,50,50,1.0
1287,2022-11-30,000016.SH,50,50,1.0
876,2020-01-23,000050.SH,50,50,1.0
486,2017-05-31,399673.SZ,50,50,1.0
1574,2024-07-31,000050.SH,50,50,1.0
488,2017-06-30,000015.SH,50,50,1.0
489,2017-06-30,000016.SH,50,50,1.0
490,2017-06-30,000050.SH,50,50,1.0
1573,2024-07-31,000016.SH,50,50,1.0
1576,2024-07-31,000688.SH,50,50,1.0


In [56]:
index_check = (
    check
    .groupby("index_code", as_index=False)
    .agg(
        n_dates=("date", "nunique"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        min_stock_n=("stock_n", "min"),
        max_stock_n=("stock_n", "max"),
        min_weight_sum=("weight_sum", "min"),
        max_weight_sum=("weight_sum", "max"),
    )
)

display(index_check.sort_values("index_code"))

,index_code,n_dates,min_date,max_date,min_stock_n,max_stock_n,min_weight_sum,max_weight_sum
0,000010.SH,147,2014-01-30,2026-03-31,180,180,1.0,1.0
1,000015.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
2,000016.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
3,000050.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
4,000300.SH,124,2016-01-29,2026-04-17,300,300,1.0,1.0
5,000510.CSI,18,2024-10-31,2026-03-31,500,500,1.0,1.0
6,000688.SH,69,2020-07-31,2026-03-31,50,50,1.0,1.0
7,000698.SH,32,2023-08-31,2026-03-31,100,100,1.0,1.0
8,000852.SH,131,2015-05-29,2026-03-31,1000,1002,1.0,1.0
9,000905.SH,147,2014-01-30,2026-03-31,500,500,1.0,1.0


In [57]:
index_check = (
    check
    .groupby("index_code", as_index=False)
    .agg(
        n_dates=("date", "nunique"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        min_stock_n=("stock_n", "min"),
        max_stock_n=("stock_n", "max"),
        min_weight_sum=("weight_sum", "min"),
        max_weight_sum=("weight_sum", "max"),
    )
)

display(index_check.sort_values("index_code"))

,index_code,n_dates,min_date,max_date,min_stock_n,max_stock_n,min_weight_sum,max_weight_sum
0,000010.SH,147,2014-01-30,2026-03-31,180,180,1.0,1.0
1,000015.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
2,000016.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
3,000050.SH,147,2014-01-30,2026-03-31,50,50,1.0,1.0
4,000300.SH,124,2016-01-29,2026-04-17,300,300,1.0,1.0
5,000510.CSI,18,2024-10-31,2026-03-31,500,500,1.0,1.0
6,000688.SH,69,2020-07-31,2026-03-31,50,50,1.0,1.0
7,000698.SH,32,2023-08-31,2026-03-31,100,100,1.0,1.0
8,000852.SH,131,2015-05-29,2026-03-31,1000,1002,1.0,1.0
9,000905.SH,147,2014-01-30,2026-03-31,500,500,1.0,1.0


In [58]:
readiness = index_check.copy()
readiness["data_file"] = str(RAW / "index_constituents.parquet")
readiness["source"] = "tushare.index_weight"
readiness["weight_normalized"] = True
readiness["known_limitations"] = ""

readiness.loc[readiness["index_code"].eq("000300.SH"), "known_limitations"] = "Tushare only covers from 2016-01-29 in current pull"
readiness.loc[readiness["index_code"].eq("000922.SH"), "known_limitations"] = "Tushare only covers to 2018-11-30 in current pull"
readiness.loc[readiness["index_code"].eq("899050.BJ"), "known_limitations"] = "Some snapshots have 100 constituents; needs iFinD/manual verification"
readiness.loc[readiness["index_code"].eq("000852.SH"), "known_limitations"] = "Early incomplete snapshots removed; starts from 2015-05-29"

readiness_path = RAW / "index_constituents_readiness_report.csv"
readiness.to_csv(readiness_path, index=False, encoding="utf-8-sig")
print("saved:", readiness_path)

saved: /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/index_constituents_readiness_report.csv


In [59]:
test = pd.read_parquet(RAW / "index_constituents.parquet")
print(test.shape)
print(test.dtypes)
display(test.head())

(1001028, 7)
date                datetime64[us]
index_code                  object
code                        object
weight                     float64
source                      object
snapshot_quality            object
weight_raw                 float64
dtype: object


,date,index_code,code,weight,source,snapshot_quality,weight_raw
0,2014-01-30,000010.SH,601318.SH,0.0550,tushare.index_weight,historical_weight,550.0
1,2014-01-30,000010.SH,600036.SH,0.0502,tushare.index_weight,historical_weight,502.0
2,2014-01-30,000010.SH,600016.SH,0.0486,tushare.index_weight,historical_weight,486.0
3,2014-01-30,000010.SH,601166.SH,0.0314,tushare.index_weight,historical_weight,314.0
4,2014-01-30,000010.SH,600000.SH,0.0301,tushare.index_weight,historical_weight,301.0


In [ ]:
canonical = RAW / "index_constituents.parquet"

print("canonical exists:", canonical.exists())
print("canonical size MB:", canonical.stat().st_size / 1024 / 1024)

df_check = duckdb.sql(f"""
    select
        count(*) as rows,
        count(distinct index_code) as n_index,
        min(date) as min_date,
        max(date) as max_date
    from read_parquet('{str(canonical).replace("'", "''")}')
""").df()

display(df_check)

weight_check = duckdb.sql(f"""
    select
        index_code,
        count(distinct date) as n_dates,
        min(date) as min_date,
        max(date) as max_date,
        min(weight_sum) as min_weight_sum,
        max(weight_sum) as max_weight_sum
    from (
        select date, index_code, sum(weight) as weight_sum
        from read_parquet('{str(canonical).replace("'", "''")}')
        group by date, index_code
    )
    group by index_codecanonical = RAW / "index_constituents.parquet"
q = str(canonical).replace("'", "''")

summary = duckdb.sql(f"""
    select
        count(*) as rows,
        count(distinct index_code) as n_index,
        min(date) as min_date,
        max(date) as max_date
    from read_parquet('{q}')
""").df()

display(summary)

weight_check = duckdb.sql(f"""
    select
        index_code,
        count(distinct date) as n_dates,
        min(date) as min_date,
        max(date) as max_date,
        min(stock_n) as min_stock_n,
        max(stock_n) as max_stock_n,
        min(weight_sum) as min_weight_sum,
        max(weight_sum) as max_weight_sum
    from (
        select
            date,
            index_code,
            count(distinct code) as stock_n,
            sum(weight) as weight_sum
        from read_parquet('{q}')
        group by date, index_code
    )
    group by index_code
    order by index_code
""").df()

display(weight_check)
    order by index_code
""").df()

display(weight_check)

canonical exists: True
canonical size MB: 0.04826831817626953


,rows,n_index,min_date,max_date
0,5812,0,NaT,NaT


,index_code,n_dates,min_date,max_date,min_weight_sum,max_weight_sum
0,<NA>,0,NaT,NaT,44603.682,44603.682


## 10. 结论模板

运行完后按下面逻辑判断：

- 如果只有 `latest_only`：只能作为临时近似，不能严格复现历史指数风格暴露。
- 如果有 `historical_weight` 或 `historical_weight_manual`，并且月末覆盖完整：可以作为正式候选。
- 如果 AkShare 抓不到某些指数，通常原因是中证官网未覆盖、指数代码后缀不匹配、或该指数属于上交所/深交所/北交所非中证指数。
- 对 `000016.SH`、`000010.SH`、`000015.SH` 等上证指数，AkShare 中证接口可能失败，需要交易所、Tushare 或手动 CSV。
